In [6]:
import pandas as pd

In [7]:
# import customers
def extract_customers(path="olist_customers_dataset.csv"):
    customers = pd.read_csv(path)
    print(f"Customers extracted: {customers.shape}")
    return customers

In [8]:
if __name__ == "__main__":
    extract_customers()

Customers extracted: (99441, 5)


In [9]:
# import products
def extract_products(path="olist_products_dataset.xlsx"):
    products = pd.read_excel(path)
    print(f"Products extracted: {products.shape}")
    return products

In [10]:
if __name__ == "__main__":
    extract_products()

Products extracted: (32951, 9)


In [11]:
import requests

In [12]:
# import engagement
def extract_engagement(url="https://fakestoreapi.com/carts"):
  response = requests.get(url)
  data = response.json()
  engagement = pd.DataFrame(data)
  print(f"Engagement extracted: {engagement.shape}")
  return engagement

In [13]:
if __name__ == "__main__":
  extract_engagement()

Engagement extracted: (7, 5)


In [14]:
# clean customers

def clean_customers(df):
  df.columns = df.columns.str.lower().str.replace(" ", "_")

  df["customer_id"] = df["customer_id"].astype(str)
  df["customer_unique_id"] = df["customer_unique_id"].astype(str)

  df.fillna("unknown", inplace=True)

  df.drop_duplicates(subset="customer_id", inplace=True)

  print(f"Customers cleaned.")
  return df


In [15]:
import os

if __name__ == "__main__":
    raw = extract_customers()  # Use the defined function to extract customers without an explicit path
    clean = clean_customers(raw)

    # Ensure the output directory exists
    output_dir = "data/clean"
    os.makedirs(output_dir, exist_ok=True)

    clean.to_csv(os.path.join(output_dir, "customers_clean.csv"), index=False)

Customers extracted: (99441, 5)
Customers cleaned.


In [16]:
def clean_products(df):
  df.columns = df.columns.str.lower().str.replace(" ", "_")

  df["product_id"] = df["product_id"].astype(str)

  numeric_cols = df.select_dtypes(include=["float64","int64"]).columns
  df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

  df.drop_duplicates(subset="product_id", inplace=True)

  print(f"Products cleaned.")
  return df

In [17]:
import os

if __name__ == "__main__":
    # The file 'data/raw/products.xlsx' does not exist.
    # Instead, use the extract_products function which correctly reads 'olist_products_dataset.xlsx'.
    raw = extract_products()
    clean = clean_products(raw)

    # Ensure the output directory exists
    output_dir = "data/clean"
    os.makedirs(output_dir, exist_ok=True)

    clean.to_csv(os.path.join(output_dir, "products_clean.csv"), index=False)

Products extracted: (32951, 9)
Products cleaned.


In [18]:
import ast # Import ast module for safe evaluation of string literals

# clean engagement

def clean_engagement(df):
  df.columns = df.columns.str.lower().str.replace(" ", "_")

  if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

  # Process the 'products' column to extract product_id
  if 'products' in df.columns:
    # Convert string representation of list to actual list of dictionaries
    # Use ast.literal_eval for safe evaluation of string literals
    df['parsed_products'] = df['products'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    # Extract the first productId from the list, or None if list is empty/missing
    # Assuming for simplicity that we only care about the first product_id if multiple exist.
    df['product_id'] = df['parsed_products'].apply(
        lambda x: x[0]['productId'] if isinstance(x, list) and len(x) > 0 and 'productId' in x[0] else None
    )
    # Drop the intermediate 'parsed_products' and original 'products' column
    df.drop(columns=['products', 'parsed_products'], inplace=True, errors='ignore')

  df.fillna("unknown", inplace=True)

  print(f"Engagement cleaned.")
  return df

In [19]:
import os

if __name__ == "__main__":
    # The file 'data/raw/engagement.json' does not exist.
    # Use the extract_engagement function which fetches data from the API.
    raw = extract_engagement()
    clean = clean_engagement(raw)

    # Ensure the output directory exists
    output_dir = "data/clean"
    os.makedirs(output_dir, exist_ok=True)

    clean.to_csv(os.path.join(output_dir, "engagement_clean.csv"), index=False)

Engagement extracted: (7, 5)
Engagement cleaned.


In [20]:
# load scripts (SQL)

import sqlite3

In [21]:
import os

def load_to_sql():
  # Ensure the directory for the database exists
  db_dir = "data/final"
  os.makedirs(db_dir, exist_ok=True)
  conn = sqlite3.connect(os.path.join(db_dir, "master.db"))

  customers = pd.read_csv("data/clean/customers_clean.csv")
  products = pd.read_csv("data/clean/products_clean.csv")
  engagement = pd.read_csv("data/clean/engagement_clean.csv")

  customers.to_sql("customers", conn, if_exists="replace", index=False)
  products.to_sql("products", conn, if_exists="replace", index=False)
  engagement.to_sql("engagement", conn, if_exists="replace", index=False)

  print("Data loaded to SQL.")
  return conn

In [22]:
if __name__ == "__main__":
    load_to_sql()

Data loaded to SQL.


In [23]:
query = """
SELECT
  e.id AS engagement_id,
  e.userid AS customer_id,
  p.product_id,
  p.product_category_name,
  c.customer_city,
  c.customer_state
FROM engagement e
LEFT JOIN customers c ON e.userid = c.customer_id
LEFT JOIN products p ON e.product_id = p.product_id
"""

In [24]:
import sqlite3
import pandas as pd

# export master dataset

def export_master():
  conn = sqlite3.connect("data/final/master.db")
  # Use the globally defined 'query' variable instead of reading from a file
  master = pd.read_sql_query(query, conn)
  master.to_csv("data/final/master_dataset.csv", index=False)
  print("Master dataset exported.")

In [25]:
if __name__ == "__main__":
    export_master()

Master dataset exported.
